In [1]:
import os, sys

base_dir = None
try:
    base_dir = os.path.dirname(os.path.abspath(__file__))
except NameError:
    base_dir = os.getcwd()

sys.path.append(
    os.path.normpath(
        os.path.join(base_dir, "..", "janic")
    )
)

from unDataStream import DataRepository, ResolutionQueryEngine  # type: ignore

In [2]:
from pathlib import Path

repo = DataRepository(
    config_path=os.path.normpath(
        os.path.join(
            base_dir,
            "..",
            "janic",
            "config",
            "data_sources.yaml",
        )
    )
)
query_engine = ResolutionQueryEngine(repo)

INFO - Logging setup complete.
INFO - Initializing UNDataRepository
INFO - URL is valid: https://digitallibrary.un.org/nanna/record/4075456/files/unbist-20250708_2.ttl?withWatermark=0&withMetadata=0&registerDownload=1&version=1
INFO - Cached Data files found.


e:\ETH\UN\policy-pulse-old\notebooks\janic\unDataStream\data\repository.py:157: DtypeWarning: Columns (67,68,169,179) have mixed types. Specify dtype option on import or set low_memory=False.
  self.resolution_table = pd.read_csv(data_path / 'resolution_table.csv')


INFO - Cached data loaded successfully.
INFO - Initialization Complete with Cached Data.


In [3]:
data_all = query_engine.query_resolutions(
    # start_date=f'{2025}-01-01',
    # end_date=f'{2025}-12-31',
)
print(f"Number of Loaded Resolutions: {len(data_all)}")

INFO - 
Final result: 5534 resolutions
Number of Loaded Resolutions: 5534


In [4]:
print(data_all.head())
print(data_all.tail())
import pandas as pd
# Find the earliest year within the data
earliest_year = pd.to_datetime(data_all['date'], errors='coerce').dt.year.min()
latest_year = pd.to_datetime(data_all['date'], errors='coerce').dt.year.max()
print(f"Earliest year in the data: {earliest_year}")
print(f"Latest year in the data: {latest_year}")


   undl_id       date session   resolution                      draft  \
0   278340 1983-10-27      38   A/RES/38/3    A/38/L.2|A/38/L.2/Add.1   
1   278341 1983-11-02      38   A/RES/38/7    A/38/L.8|A/38/L.8/Add.1   
2   278342 1983-11-10      38   A/RES/38/9             A/38/L.7/Rev.2   
3   278343 1983-11-15      38  A/RES/38/11  A/38/L.15|A/38/L.15/Add.1   
4   278344 1983-11-16      38  A/RES/38/12                  A/38/L.12   

  committee_report     meeting  \
0              NaN  A/38/PV.38   
1              NaN  A/38/PV.43   
2              NaN  A/38/PV.52   
3              NaN  A/38/PV.56   
4              NaN  A/38/PV.59   

                                               title  \
0  The situation in Kampuchea : resolution / adop...   
1  The situation in Grenada : resolution / adopte...   
2  Armed Israeli aggression against the Iraqi nuc...   
3  Proposed new racial constitution of South Afri...   
4  Question of the Falkland Islands (Malvinas) : ...   

                   

In [5]:
combined_title = data_all['title'] # + '. ' + data_all['agenda_title']

In [ ]:
from transformers import T5Tokenizer, T5ForConditionalGeneration
import torch

model = T5ForConditionalGeneration.from_pretrained("Voicelab/vlt5-base-keywords")
tokenizer = T5Tokenizer.from_pretrained("Voicelab/vlt5-base-keywords")

d:\Programs\miniconda3\envs\quantityapp\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
d:\Programs\miniconda3\envs\quantityapp\lib\site-packages\transformers\utils\hub.py:110: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(
You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565


In [7]:
import torch
print("=== 设备信息 ===")
print(f"CUDA可用: {torch.cuda.is_available()}")
print(f"CUDA设备数量: {torch.cuda.device_count()}")
print(f"当前设备: {torch.cuda.current_device() if torch.cuda.is_available() else 'CPU'}")

=== 设备信息 ===
CUDA可用: True
CUDA设备数量: 1
当前设备: 0


In [11]:
model.to('cuda')
# tokenizer.to('cuda')
print(f"模型设备: {next(model.parameters()).device}")

模型设备: cuda:0


In [16]:
task_prefix = "Keywords: "
keywords_list = []

window_size = 24
stride = 12

for idx, sample in enumerate(combined_title):
    words = str(sample).split()
    print(f"Process {idx} / {len(combined_title)}, 文本长度: {len(words)} 词")
    keywords_set = set()
    
    if len(words) <= window_size:
        # 短文本直接处理
        input_sequence = task_prefix + " ".join(words)
        input_ids = tokenizer([input_sequence], return_tensors="pt", max_length=512, truncation=True).input_ids
        # print(input_ids.shape)
        input_ids = input_ids.to('cuda')
        output = model.generate(input_ids, no_repeat_ngram_size=3, num_beams=4, max_length=100)
        predicted = tokenizer.decode(output[0], skip_special_tokens=True)
        # 更稳健的分割方式
        keywords = [kw.strip() for kw in predicted.split(",") if kw.strip()]
        keywords_set.update(keywords)
    else:
        # 长文本滑动窗口处理
        for start in range(0, len(words), stride):
            end_idx = min(start + window_size, len(words))
            window = words[start:end_idx]
            if len(window) < 5:  # 跳过太短的窗口
                continue
                
            input_sequence = task_prefix + " ".join(window)
            input_ids = tokenizer([input_sequence], return_tensors="pt", max_length=512, truncation=True).input_ids
            input_ids = input_ids.to('cuda')
            output = model.generate(input_ids, no_repeat_ngram_size=3, num_beams=4, max_length=100)
            predicted = tokenizer.decode(output[0], skip_special_tokens=True)
            keywords = [kw.strip() for kw in predicted.split(",") if kw.strip()]
            keywords_set.update(keywords)
    
    # 过滤掉太短的关键词
    filtered_keywords = [kw for kw in keywords_set if len(kw) >= 3]
    final_result = ", ".join(sorted(filtered_keywords))
    keywords_list.append(final_result)
    print(f"原文: {sample}")
    print(f"关键词: {final_result}")
    print("-" * 50)

Process 0 / 5534, 文本长度: 12 词
原文: The situation in Kampuchea : resolution / adopted by the General Assembly
关键词: Kampuchea, resolution
--------------------------------------------------
Process 1 / 5534, 文本长度: 12 词
原文: The situation in Grenada : resolution / adopted by the General Assembly
关键词: Grenada, resolution, situation
--------------------------------------------------
Process 2 / 5534, 文本长度: 42 词
原文: Armed Israeli aggression against the Iraqi nuclear installations and its grave consequences for the established international system concerning the peaceful uses of nuclear energy, the non-proliferation of nuclear weapons and international peace and security : resolution / adopted by the General Assembly
关键词: General Assembly, Iraqi nuclear installations, Israeli aggression, international peace and security, non-proliferation, nuclear energy, nuclear weapon, resolution, security
--------------------------------------------------
Process 3 / 5534, 文本长度: 15 词
原文: Proposed new racial co

In [19]:
# 将关键词列表保存到 CSV 文件
# time_str = get_time_string()
df_keywords = pd.DataFrame({
    'undl_id': data_all['undl_id'],
    'keywords': keywords_list
})
df_keywords.to_csv(f"wc_data/undlid_keywords.csv", index=False)


In [45]:
cleanCombinedTitle = pd.read_csv(f"wc_data/undlid_cleanCombinedTitle_{time_str}.csv")

In [46]:
data_filtered = pd.merge(data_filtered, cleanCombinedTitle, on='undl_id', how='left')
print("data_filtered columns: ", data_filtered.columns)

data_filtered columns:  Index(['undl_id', 'date', 'session', 'resolution', 'draft', 'committee_report',
       'meeting', 'title', 'agenda_title', 'total_yes',
       ...
       'YMD', 'YUG', 'ZAF', 'ZMB', 'ZWE', 'title_clean', 'combined_title_x',
       'combined_title_y', 'combined_title', 'wc_title'],
      dtype='object', length=222)


In [48]:
print("data_filtered.head(): \n", data_filtered[['undl_id', 'combined_title', 'wc_title']].head())

data_filtered.head(): 
    undl_id                                     combined_title  \
0   278340                                          kampuchea   
1   278341                                            grenada   
2   278342  armed israeli aggression against the iraqi nuc...   
3   278343   proposed new racial constitution of south africa   
4   278344                    the falkland islands (malvinas)   

                                            wc_title  
0                                          kampuchea  
1                                            grenada  
2  armed israeli aggression against the iraqi nuc...  
3   proposed new racial constitution of south africa  
4                    the falkland islands (malvinas)  
